In [5]:
import subprocess  # Modul untuk menjalankan file atau perintah lain dari Python
import sys         # Modul untuk mengakses fungsi sistem (seperti path python)

# Membuat teks kode python yang akan disimpan ke file terpisah
skrip_self_deadlock = "\n".join([
    "import threading",                                    # Memasukkan modul threading ke dalam skrip
    "",
    "lock = threading.Lock()",                             # Bikin kunci (lock) biasa
    "",
    "def rekursif(n):",                                    # Fungsi yang manggil dirinya sendiri (rekursif)
    "    with lock:",                                      # Ambil kunci sebelum lanjut
    "        print(f'masuk level {n}')",                   # Cetak posisi level saat ini
    "        if n > 0:",                                   # Kalau n masih di atas 0...
    "            rekursif(n - 1)  # mencoba ambil lock yang SAMA lagi!", # Panggil diri sendiri tapi coba ambil kunci yang sama lagi!
    "",
    "t = threading.Thread(target=rekursif, args=(2,))",     # Bikin thread baru buat jalanin fungsi di atas
    "t.start()",                                           # Mulai jalankan thread-nya
    "t.join()",                                            # Tunggu thread selesai (di sini nanti bakal macet/deadlock)
    "print('selesai tanpa deadlock')"                      # Bagian ini tidak bakal pernah nyampe
])

# Menyimpan teks di atas ke dalam file sementara bernama demo_self_deadlock.py
with open("demo_self_deadlock.py", "w") as f:
    f.write(skrip_self_deadlock)

try:
    # Menjalankan file python yang baru dibuat tadi, batasi waktu maksimal 3 detik
    hasil = subprocess.run(
        [sys.executable, "demo_self_deadlock.py"],
        capture_output=True, text=True, timeout=3
    )
    print("Output:", hasil.stdout)
except subprocess.TimeoutExpired as e:
    # Kalau prosesnya lebih dari 3 detik (artinya macet/deadlock), jalankan kode ini
    print("TIMEOUT setelah 3 detik -- SELF-DEADLOCK TERKONFIRMASI!")
    
    # PERBAIKAN: Langsung pakai e.stdout tanpa .decode() karena sudah berupa string
    print("Output sebelum macet:", e.stdout if e.stdout else "(kosong)")
    
    # Penjelasan kenapa bisa macet
    print("\nPenjelasan: Thread mau ngunci `lock` yang SAMA untuk kedua kalinya")
    print("padahal lock-nya belum dilepas. Karena Lock biasa tidak bisa dikunci")
    print("dua kali oleh orang yang sama, akhirnya thread nungguin dirinya sendiri")
    print("sampai selamanya (macet total).")

TIMEOUT setelah 3 detik -- SELF-DEADLOCK TERKONFIRMASI!
Output sebelum macet: masuk level 2


Penjelasan: Thread mau ngunci `lock` yang SAMA untuk kedua kalinya
padahal lock-nya belum dilepas. Karena Lock biasa tidak bisa dikunci
dua kali oleh orang yang sama, akhirnya thread nungguin dirinya sendiri
sampai selamanya (macet total).


In [8]:
import threading  # Modul untuk mengelola banyak proses secara bersamaan (multithreading)

lock = threading.RLock()  # Bikin RLock khusus yang bisa dikunci berkali-kali oleh thread yang sama tanpa macet

def rekursif(n): # membuat fungsi rekursif
    with lock:  # Mengambil kunci RLock sebelum lanjut 
        print(f"masuk level{n}")  # Mencetak posisi level rekursi saat ini
        if n>0:  # Jika nilai n masih di atas 0...
            rekursif(n-1)  # Panggil fungsi ini lagi dengan n dikurangi 1 

t = threading.Thread(target=rekursif, args=(2,))  # Membuat thread baru buat menjalankan fungsi rekursif dengan angka awal 2
t.start()  # Memulai jalannya thread
t.join()  # Menunggu sampai thread-nya benar-benar selesai
print("selesai tanpa deadlock (berkat RLock)")  # Pesan sukses tercetak karena program tidak mengalami macet/deadlock

masuk level2
masuk level1
masuk level0
selesai tanpa deadlock (berkat RLock)


In [15]:
import threading  # Modul untuk menjalankan banyak proses (thread) secara bersamaan
import time       # Modul untuk memberikan jeda waktu (delay) pada program

sem = threading.Semaphore(1)  # Membuat Semaphore dengan nilai 1, yang berfungsi mengatur batasan berapa thread boleh masuk bersamaan

def gunakan_slot(nama):
    print(f"[{nama}] menunggu slot...")  # Menampilkan pesan bahwa thread sedang antre meminta izin masuk
    with sem:  # Meminta izin ke Semaphore. Kalau pintu penuh, thread akan menunggu di sini sampai ada yang keluar
        print(f"[{nama}] MENDAPAT slot, bekerja selama 0.3 detik...")  # Pesan saat thread berhasil masuk dan mulai bekerja
        time.sleep(0.3)  # Mensimulasikan proses pekerjaan yang memakan waktu 0.3 detik
        print(f"[{nama}] selesai, melepas slot")  # Pesan saat thread selesai dan mengembalikan kunci/slot ke Semaphore

threads = [threading.Thread(target=gunakan_slot, args=(f"T{i}",))for i in range(5)]  # Membuat 5 buah thread (T0 sampai T4) yang semuanya mau menjalankan fungsi gunakan_slot
for t in threads:
    t.start()  # Menyalakan/memulai semua thread secara bersamaan (berebut masuk ke Semaphore)
for t in threads:
    t.join()   # Menunggu sampai semua thread benar-benar selesai bekerja

print("\nPerhatikan: meski ada 5 thread, HANYA 2 yang bekerja bersamaan pada satu waktu.")  # Catatan/kesimpulan program (Catatan: karena nilai semaphore di atas diset 1, sebenarnya hanya 1 thread yang masuk pada satu waktu).

[T0] menunggu slot...
[T0] MENDAPAT slot, bekerja selama 0.3 detik...
[T1] menunggu slot...
[T2] menunggu slot...
[T3] menunggu slot...
[T4] menunggu slot...
[T0] selesai, melepas slot
[T1] MENDAPAT slot, bekerja selama 0.3 detik...
[T1] selesai, melepas slot
[T2] MENDAPAT slot, bekerja selama 0.3 detik...
[T2] selesai, melepas slot
[T3] MENDAPAT slot, bekerja selama 0.3 detik...
[T3] selesai, melepas slot
[T4] MENDAPAT slot, bekerja selama 0.3 detik...
[T4] selesai, melepas slot

Perhatikan: meski ada 5 thread, HANYA 2 yang bekerja bersamaan pada satu waktu.


In [21]:
import threading  # Modul untuk mengelola banyak proses (thread) secara bersamaan

sem = threading.Semaphore(2)  # Membuat Semaphore biasa dengan batas maksimal 2 izin
sem.release()  # Sengaja menambah izin semaphore tanpa alasan yang jelas sebanyak 1 kali
sem.release()  # Sengaja merilis izin semaphore lagi untuk kedua kalinya
print("Semaphore biasa: rilis berlebihan TIDAK memunculkan error (bug tersembunyi)")  # Pesan yang menunjukkan sifat Semaphore biasa yang longgar (tidak protes meski dirilis berlebihan)

bsem = threading.BoundedSemaphore(2)  # Membuat BoundedSemaphore (Semaphore terbatas) dengan batas maksimal 2 izin
try:
    bsem.release()  # Sengaja merilis BoundedSemaphore yang belum pernah dipakai 
    print("ini seharusnya tidak pernah tercetak")  # Pesan ini dilewati karena baris di atas keburu error
except ValueError as e:
    print(f"BoundedSemaphore MENDETEKSI kesalahan: {e}")  # Menangkap error ValueError dan mencetak pesan bahwa BoundedSemaphore berhasil mendeteksi kesalahan rilis berlebihan

Semaphore biasa: rilis berlebihan TIDAK memunculkan error (bug tersembunyi)
BoundedSemaphore MENDETEKSI kesalahan: Semaphore released too many times


In [22]:
import threading  # Modul untuk menjalankan banyak proses (thread) secara bersamaan
import time       # Modul untuk memberikan jeda waktu (delay)

kondisi = threading.Condition()  # Membuat objek Condition untuk koordinasi antar-thread (mekanisme tunggu-menunggu)
status = {"siap": False}         # Kamus data bersama (shared data) untuk menandai status siap, awalnya False (belum siap)

def penunggu(nama):
    with kondisi:                # Mengambil kunci kondisi sebelum mengecek atau menunggu
        print(f"[{nama}] menunggu status siap...")  # Menampilkan pesan bahwa thread sedang bersiap menunggu
        # wait_for() melepas lock SEMENTARA menunggu, dan mengambilnya lagi
        # otomatis begitu predicate bernilai True DAN mendapat notify
        kondisi.wait_for(lambda: status["siap"])  # Thread akan tidur di sini sampai status["siap"] menjadi True
        print(f"[{nama}] status sudah siap, lanjut kerja!")  # Pesan ketika thread berhasil bangun dan melanjutkan tugas

def pemberi_sinyal():
    time.sleep(0.5)              # Menunggu selama 0.5 detik sebelum memberikan sinyal
    with kondisi:                # Mengambil kunci kondisi untuk mengubah status bersama
        print("[Pemberi sinyal] mengubah status jadi siap dan notify_all()")  # Info bahwa sinyal akan dikirim
        status["siap"] = True    # Mengubah status["siap"] menjadi True
        kondisi.notify_all()     # Membangunkan SEMUA thread yang sedang menunggu (wait) agar mereka mengecek ulang kondisinya

# Membuat 3 buah thread penunggu (Penunggu-0, Penunggu-1, Penunggu-2)
threads = [threading.Thread(target=penunggu, args=(f"Penunggu-{i}",)) for i in range(3)]
# Membuat 1 thread khusus sebagai pemberi sinyal
t_sinyal = threading.Thread(target=pemberi_sinyal)

for t in threads:
    t.start()                    # Menyalakan semua thread penunggu (mereka langsung masuk antrean tunggu)
t_sinyal.start()                 # Menyalakan thread pemberi sinyal (akan menunggu 0.5 detik lalu mengubah status)
for t in threads:
    t.join()                     # Menunggu semua thread penunggu selesai bekerja
t_sinyal.join()                  # Menunggu thread pemberi sinyal selesai

[Penunggu-0] menunggu status siap...
[Penunggu-1] menunggu status siap...
[Penunggu-2] menunggu status siap...
[Pemberi sinyal] mengubah status jadi siap dan notify_all()
[Penunggu-2] status sudah siap, lanjut kerja!
[Penunggu-0] status sudah siap, lanjut kerja!
[Penunggu-1] status sudah siap, lanjut kerja!


In [24]:
import subprocess  # Modul untuk menjalankan file atau proses lain dari dalam Python
import sys         # Modul untuk mengakses fungsi sistem (seperti mengambil path interpreter python)

# Membuat teks kode python utuh yang nantinya akan disimpan ke file terpisah untuk mensimulasikan deadlock
skrip_deadlock = "\n".join([
    "import threading, time",                                  # Mengimpor modul threading dan time di dalam skrip anak
    "",
    "lock_a = threading.Lock()",                               # Membuat kunci pertama bernama lock_a
    "lock_b = threading.Lock()",                               # Membuat kunci kedua bernama lock_b
    "",
    "def thread1():",                                          # Mendefinisikan fungsi untuk thread pertama
    "    with lock_a:",                                        # Thread 1 mengunci/memegang lock_a terlebih dahulu
    "        print('thread1 pegang lock_a, tunggu 0.3s...')",  # Menampilkan pesan status
    "        time.sleep(0.3)",                                 # Jeda sejenak selama 0.3 detik
    "        print('thread1 mencoba ambil lock_b...')",        # Pesan saat thread 1 mencoba merebut lock_b
    "        with lock_b:",                                    # Thread 1 mencoba mengambil lock_b (yang sedang dipegang thread 2
    "            print('thread1 dapat keduanya!')",          # Pesan jika berhasil mendapat keduanya (tidak akan tercapai)
    "",
    "def thread2():",                                          # Mendefinisikan fungsi untuk thread kedua
    "    with lock_b:",                                        # Thread 2 mengunci/memegang lock_b terlebih dahulu
    "        print('thread2 pegang lock_b, tunggu 0.3s...')",  # Menampilkan pesan status
    "        time.sleep(0.3)",                                 # Jeda sejenak selama 0.3 detik
    "        print('thread2 mencoba ambil lock_a...')",        # Pesan saat thread 2 mencoba merebut lock_a
    "        with lock_a:",                                    # Thread 2 mencoba mengambil lock_a (yang sedang dipegang thread 1)
    "            print('thread2 dapat keduanya!')",          # Pesan jika berhasil mendapat keduanya (tidak akan tercapai)
    "",
    "t1 = threading.Thread(target=thread1)",                   # Membuat objek thread pertama yang menjalankan fungsi thread1
    "t2 = threading.Thread(target=thread2)",                   # Membuat objek thread kedua yang menjalankan fungsi thread2
    "t1.start(); t2.start()",                                  # Menyalakan kedua thread secara bersamaan
    "t1.join(); t2.join()",                                    # Menunggu kedua thread selesai (di sinilah mereka saling menunggu selamanya / deadlock
    "print('SELESAI - tidak ada deadlock')",                   # Pesan sukses yang tidak akan pernah tercetak
])

# Menyimpan teks di atas ke dalam file bernama "demo_deadlock.py"
with open("demo_deadlock.py", "w") as f:
    f.write(skrip_deadlock)  # Menulis isi kode ke dalam file

try:
    # Menjalankan file "demo_deadlock.py" sebagai proses terpisah dengan batas waktu maksimal 3 detik
    hasil = subprocess.run([sys.executable, "demo_deadlock.py"], capture_output=True, text=True, timeout=3)
    print("Output:", hasil.stdout)  # Dicetak jika proses selesai normal (tidak terjadi deadlock)
except subprocess.TimeoutExpired as e:
    # Blok ini berjalan jika proses memakan waktu lebih dari 3 detik (artinya terjadi macet total / deadlock)
    print("TIMEOUT setelah 3 detik -- DEADLOCK TERKONFIRMASI!")  # Konfirmasi bahwa deadlock terjadi
    print("Output sebelum macet total:")                        # Judul info output awal
    print(e.stdout if e.stdout else "(kosong)")                # Menampilkan teks apa saja yang sempat tercetak sebelum program macet

TIMEOUT setelah 3 detik -- DEADLOCK TERKONFIRMASI!
Output sebelum macet total:
thread1 pegang lock_a, tunggu 0.3s...
thread2 pegang lock_b, tunggu 0.3s...
thread1 mencoba ambil lock_b...
thread2 mencoba ambil lock_a...



In [ ]:
import subprocess  # Modul untuk menjalankan file atau proses lain dari dalam Python
import sys         # Modul untuk mengakses fungsi sistem (seperti mengambil path interpreter python

# Membuat teks kode python utuh yang nantinya disimpan ke file untuk mensimulasikan perbaikan deadlock
skrip_fix = "\n".join([
    "import threading, time",                                  # Mengimpor modul threading dan time di dalam skrip anak
    "",
    "lock_a = threading.Lock()",                               # Membuat kunci pertama bernama lock_a
    "lock_b = threading.Lock()",                               # Membuat kunci kedua bernama lock_b
    "",
    "def thread1():",                                          # Mendefinisikan fungsi untuk thread pertama
    "    with lock_a:",                                        # Thread 1 mengambil kunci lock_a terlebih dahulu[
    "        print('thread1 pegang lock_a, tunggu 0.3s...')",  # Menampilkan pesan status
    "        time.sleep(0.3)",                                 # Jeda sejenak selama 0.3 detik
    "        print('thread1 mencoba ambil lock_b...')",        # Pesan saat thread 1 mencoba mengambil lock_b
    "        with lock_b:",                                    # Thread 1 mengambil lock_b
    "            print('thread1 dapat keduanya!')",          # Pesan jika berhasil mendapat kedua kunci
    "",
    "def thread2():",                                          # Mendefinisikan fungsi untuk thread kedua
    "    # PERBAIKAN: urutan diseragamkan -- SELALU lock_a dulu, baru lock_b",
    "    with lock_a:",                                        # Thread 2 SEKARANG JUGA mengambil lock_a duluan (urutan disamakan dengan thread1
    "        print('thread2 pegang lock_a, tunggu 0.3s...')",  # Menampilkan pesan status
    "        time.sleep(0.3)",                                 # Jeda sejenak selama 0.3 detik
    "        print('thread2 mencoba ambil lock_b...')",        # Pesan saat thread 2 mencoba mengambil lock_b
    "        with lock_b:",                                    # Thread 2 mengambil lock_b
    "            print('thread2 dapat keduanya!')",          # Pesan jika berhasil mendapat kedua kunci
    "",
    "t1 = threading.Thread(target=thread1)",                   # Membuat objek thread pertama
    "t2 = threading.Thread(target=thread2)",                   # Membuat objek thread kedua
    "t1.start(); t2.start()",                                  # Menyalakan kedua thread secara bersamaan
    "t1.join(); t2.join()",                                    # Menunggu kedua thread selesai bekerja (karena urutannya sama, tidak ada yang saling tunggu selamanya
    "print('SELESAI - tidak ada deadlock')",                   # Pesan sukses yang berhasil tercetak
])

# Menyimpan teks kode di atas ke dalam file sementara
with open("/tmp/demo_deadlock_fixed.py", "w") as f:
    f.write(skrip_fix)  # Menulis isi string ke dalam file

# Menjalankan file python yang baru dibuat tadi sebagai subprocess dengan batas waktu 3 detik
hasil = subprocess.run([sys.executable, "/tmp/demo_deadlock_fixed.py"], capture_output=True, text=True, timeout=3)
print("Output:")       # Mencetak label output
print(hasil.stdout)    # Menampilkan hasil cetakan (stdout) dari skrip yang dijalankan
print("Kode keluar (return code):", hasil.returncode, "-- 0 berarti BERHASIL tanpa timeout/error")  # Menampilkan kode status keluar proses

Output:

Kode keluar (return code): 2 -- 0 berarti BERHASIL tanpa timeout/error


In [8]:
import threading
import time
import random

lock_a = threading.Lock()
lock_b = threading.Lock()

def kerja_aman(nama, lock_pertama, lock_kedua, max_percobaan=10):
    for percobaan in range(1, max_percobaan + 1):
        if lock_pertama.acquire(timeout=0.2):
            try:
                time.sleep(0.05)  # Simulasi kerja singkat sambil memegang lock pertama
                
                if lock_kedua.acquire(timeout=0.2):
                    try:
                        print(f"[{nama}] berhasil dapat KEDUA lock pada percobaan ke-{percobaan}")
                        return
                    finally:
                        lock_kedua.release()  # Pastikan lock kedua dilepas jika sempat didapat
                else:
                    print(f"[{nama}] gagal dapat lock kedua (percobaan {percobaan}), lepas lock pertama & coba lagi")
            
            finally:
                lock_pertama.release()  # Pastikan lock pertama selalu dilepas di akhir percobaan ini
                
        time.sleep(random.uniform(0.01, 0.05))  # Jeda acak sebelum mencoba lagi
        
    print(f"[{nama}] MENYERAH setelah {max_percobaan} percobaan")

# Thread 1 dan Thread 2 mencoba mengambil lock dengan urutan berlawanan (potensi deadlock yang dicegah dengan timeout & retry)
t1 = threading.Thread(target=kerja_aman, args=("Thread-1", lock_a, lock_b))
t2 = threading.Thread(target=kerja_aman, args=("Thread-2", lock_b, lock_a))

t1.start()
t2.start()

t1.join()
t2.join()

print("\nKeduanya selesai TANPA deadlock permanen, meski urutan pengambilan lock berlawanan!")

[Thread-1] gagal dapat lock kedua (percobaan 1), lepas lock pertama & coba lagi
[Thread-2] gagal dapat lock kedua (percobaan 1), lepas lock pertama & coba lagi
[Thread-2] gagal dapat lock kedua (percobaan 2), lepas lock pertama & coba lagi
[Thread-1] berhasil dapat KEDUA lock pada percobaan ke-2
[Thread-2] berhasil dapat KEDUA lock pada percobaan ke-3

Keduanya selesai TANPA deadlock permanen, meski urutan pengambilan lock berlawanan!


In [7]:
import threading  # Mengimpor modul untuk mengelola banyak thread
import time       # Mengimpor modul untuk memberikan jeda waktu (delay)

lock_status = threading.Lock()  # Membuat kunci untuk mengamankan pengecekan status lapar bersama
lapar = {"A": True, "B": True}  # Kamus status awal: Thread A dan B sama-sama berstatus lapar (True)

def makan_sopan(nama, lawan, max_percobaan=8):  # Fungsi simulasi "sopan" (saling mengalah) dengan batasan 8 kali percobaan
    percobaan = 0  # Menghitung jumlah percobaan yang sudah dilakukan
    while lapar[nama] and percobaan < max_percobaan:  # Perulangan selama thread masih lapar dan belum mencapai batas percobaan
        percobaan += 1  # Menambah jumlah percobaan setiap kali masuk perulangan
        with lock_status:  # Mengunci akses untuk mengecek status lawan secara aman
            if lapar[lawan]:  # Jika lawan ternyata masih lapar
                print(f"[{nama}] percobaan {percobaan}: {lawan} masih lapar, saya mengalah dulu...")  # Pesan saat mengalah
            else:  # Jika lawan sudah selesai makan
                print(f"[{nama}] percobaan {percobaan}: {lawan} sudah selesai, saya makan sekarang!")  # Pesan saat bisa makan
                lapar[nama] = False  # Mengubah status diri sendiri menjadi tidak lapar lagi (selesai)
                continue  # Lanjut ke iterasi berikutnya
        time.sleep(0.1) # Jeda sebentar; di sinilah kedua thread terus "mengalah" secara bersamaan, memicu livelock
    if lapar[nama]:  # Jika sampai batas percobaan habis statusnya masih lapar
        print(f"[{nama}] MENYERAH setelah {max_percobaan} percobaan (LIVELOCK, tidak pernah makan!)")  # Pesan tanda gagal karena livelock

# Membuat dua thread (t1 untuk A, t2 untuk B) yang saling bergantian mengalah
t1 = threading.Thread(target=makan_sopan, args=("A", "B"))
t2 = threading.Thread(target=makan_sopan, args=("B", "A"))
t1.start(); t2.start()  # Menyalakan kedua thread secara bersamaan
t1.join(); t2.join()    # Menunggu sampai kedua thread selesai atau menyerah

[A] percobaan 1: B masih lapar, saya mengalah dulu...
[B] percobaan 1: A masih lapar, saya mengalah dulu...
[B] percobaan 2: A masih lapar, saya mengalah dulu...
[A] percobaan 2: B masih lapar, saya mengalah dulu...
[B] percobaan 3: A masih lapar, saya mengalah dulu...
[A] percobaan 3: B masih lapar, saya mengalah dulu...
[B] percobaan 4: A masih lapar, saya mengalah dulu...
[A] percobaan 4: B masih lapar, saya mengalah dulu...
[B] percobaan 5: A masih lapar, saya mengalah dulu...
[A] percobaan 5: B masih lapar, saya mengalah dulu...
[B] percobaan 6: A masih lapar, saya mengalah dulu...
[A] percobaan 6: B masih lapar, saya mengalah dulu...
[B] percobaan 7: A masih lapar, saya mengalah dulu...
[A] percobaan 7: B masih lapar, saya mengalah dulu...
[B] percobaan 8: A masih lapar, saya mengalah dulu...
[A] percobaan 8: B masih lapar, saya mengalah dulu...
[B] MENYERAH setelah 8 percobaan (LIVELOCK, tidak pernah makan!)[A] MENYERAH setelah 8 percobaan (LIVELOCK, tidak pernah makan!)



In [6]:
import threading  # Mengimpor modul untuk mengelola banyak proses (thread)
import time       # Mengimpor modul untuk memberikan jeda waktu
import random     # Mengimpor modul angka acak untuk membuat pemutus simetri

lock_status = threading.Lock()  # Membuat kunci pengaman agar pengecekan status dilakukan bergantian
lapar = {"A": True, "B": True}  # Kamus status awal: Thread A dan B sama-sama berstatus lapar

def makan_sopan_v2(nama, lawan, max_percobaan=8):  # Fungsi versi perbaikan livelock dengan batas 8 kali percobaan
    percobaan = 0  # Variabel penghitung percobaan
    while lapar[nama] and percobaan < max_percobaan:  # Perulangan selama thread masih lapar dan belum mencapai batas
        percobaan += 1  # Menambah hitungan percobaan
        with lock_status:  # Mengunci akses untuk membaca/mengubah status bersama
            # Pemutus simetri acak: 70% peluang mengalah jika lawan lapar, 30% peluang nekat makan
            if lapar[lawan] and random.random() > 0.3:
                print(f"[{nama}] percobaan {percobaan}: {lawan} masih lapar, saya mengalah dulu...")
            else:  # Jika kebagian jatah nekat atau lawan sudah selesai
                print(f"[{nama}] percobaan {percobaan}: saya makan sekarang!")
                lapar[nama] = False  # Mengubah status diri sendiri menjadi tidak lapar (selesai)
                continue  # Lanjut ke iterasi berikutnya
        time.sleep(0.05)  # Jeda sebentar sebelum mencoba lagi
    if lapar[nama]:  # Jika sampai batas percobaan habis masih berstatus lapar
        print(f"[{nama}] MENYERAH setelah {max_percobaan} percobaan")

random.seed(3)  # Mengunci pola angka acak agar hasil simulasinya konsisten setiap kali dijalankan
lapar["A"], lapar["B"] = True, True  # Mengatur ulang status keduanya menjadi lapar

# Membuat dua thread untuk A dan B menggunakan fungsi versi 2
t1 = threading.Thread(target=makan_sopan_v2, args=("A", "B"))
t2 = threading.Thread(target=makan_sopan_v2, args=("B", "A"))

t1.start(); t2.start()  # Memulai jalannya kedua thread secara bersamaan
t1.join(); t2.join()    # Menunggu sampai kedua thread selesai bekerja
print("\nDengan pemutus simetri, kebuntuan berhasil dipecahkan.")  # Pesan sukses

[A] percobaan 1: saya makan sekarang!
[B] percobaan 1: saya makan sekarang!

Dengan pemutus simetri, kebuntuan berhasil dipecahkan.


In [5]:
import threading  # Modul untuk mengelola banyak proses (thread)
import time       # Modul untuk memberikan jeda waktu

lock = threading.Lock()  # Membuat kunci bersama yang akan diperebutkan oleh kedua thread
hitungan = {"Rakus": 0, "Sabar": 0}  # Kamus untuk menghitung seberapa sering masing-masing thread mendapat kunci
berhenti = threading.Event()  # Sinyal pengontrol untuk menghentikan perulangan thread nanti

def thread_rakus():
    while not berhenti.is_set():  # Terus berjalan selama sinyal berhenti belum aktif
        with lock:  # Mengambil kunci
            hitungan["Rakus"] += 1  # Menambah jumlah perolehan kunci
            # TIDAK ada jeda -> langsung mencoba lagi, hampir selalu menang rebutan lock

def thread_sabar():
    while not berhenti.is_set():  # Terus berjalan selama belum disuruh berhenti
        time.sleep(0.01)  # Memberi jeda kecil (tidur 0.01 detik) sebelum mencoba lagi agar lebih "sopan"
        with lock:  # Mengambil kunci
            hitungan["Sabar"] += 1  # Menambah jumlah perolehan kunci

t1 = threading.Thread(target=thread_rakus)  # Membuat thread rakus
t2 = threading.Thread(target=thread_sabar)  # Membuat thread sabar
t1.start(); t2.start()  # Menyalakan kedua thread secara bersamaan
time.sleep(1.0)  # Membiarkan keduanya berjalan berebut kunci selama 1 detik penuh
berhenti.set()  # Memberi sinyal agar kedua thread berhenti
t1.join(); t2.join()  # Menunggu sampai kedua thread benar-benar berhenti

# Menampilkan hasil statistik perolehan kunci ke layar
print(f"Jumlah lock didapat 'Rakus' : {hitungan['Rakus']:,}")
print(f"Jumlah lock didapat 'Sabar' : {hitungan['Sabar']:,}")
print(f"\nRasio ketimpangan: 'Rakus' mendapat lock {hitungan['Rakus']/max(hitungan['Sabar'],1):.0f}x lebih sering")
print("Ini adalah STARVATION -- 'Sabar' secara teknis TIDAK deadlock (masih dapat giliran sesekali),")
print("tapi porsinya sangat tidak adil dibanding 'Rakus'.")

Jumlah lock didapat 'Rakus' : 5,216,735
Jumlah lock didapat 'Sabar' : 45

Rasio ketimpangan: 'Rakus' mendapat lock 115927x lebih sering
Ini adalah STARVATION -- 'Sabar' secara teknis TIDAK deadlock (masih dapat giliran sesekali),
tapi porsinya sangat tidak adil dibanding 'Rakus'.


In [4]:
import threading  # Modul untuk mengelola banyak proses (thread)
import time       # Modul untuk memberikan jeda waktu
import random     # Modul untuk membuat variasi waktu acak

UKURAN_BUFFER = 3  # Kapasitas maksimal penampungan (buffer) adalah 3 item
buffer = []        # Wadah/antrean tempat penyimpanan barang (list kosong)

mutex = threading.Lock()                      # Kunci pengaman agar akses ke buffer tidak bertabrakan
slot_kosong = threading.Semaphore(UKURAN_BUFFER)  # Semaphore untuk menghitung slot kosong (mulai dari 3)
slot_terisi = threading.Semaphore(0)              # Semaphore untuk menghitung item yang siap diambil (mulai dari 0)

def producer(jumlah_item):
    for i in range(jumlah_item):
        item = f"item-{i}"  # Membuat nama item baru
        time.sleep(random.uniform(0.02, 0.08))  # Mensimulasikan proses pembuatan item
        slot_kosong.acquire()  # Menunggu sampai ada slot kosong; kalau penuh, produsen berhenti sejenak
        with mutex:            # Mengunci buffer supaya aman saat diubah
            buffer.append(item)  # Memasukkan item baru ke dalam buffer
            print(f"[Producer] taruh {item:<8} (buffer: {len(buffer)}/{UKURAN_BUFFER})")  # Menampilkan status
        slot_terisi.release()  # Memberitahu konsumen bahwa ada 1 item baru yang siap diambil

def consumer(jumlah_item, nama):
    for _ in range(jumlah_item):
        slot_terisi.acquire()  # Menunggu sampai ada item di buffer; kalau kosong, konsumen antre menunggu
        with mutex:            # Mengunci buffer saat mengambil barang
            item = buffer.pop(0)  # Mengambil item pertama dari buffer
            print(f"    [{nama}] ambil {item:<8} (buffer: {len(buffer)}/{UKURAN_BUFFER})")  # Menampilkan status
        slot_kosong.release()  # Memberitahu produsen bahwa ada 1 slot kosong baru yang bisa diisi

JUMLAH_ITEM = 9  # Total item yang akan diproduksi
t_p = threading.Thread(target=producer, args=(JUMLAH_ITEM,))  # Membuat thread untuk produsen
t_c1 = threading.Thread(target=consumer, args=(JUMLAH_ITEM // 2 + JUMLAH_ITEM % 2, "Consumer-1"))  # Thread konsumen pertama (mengambil 5 item)
t_c2 = threading.Thread(target=consumer, args=(JUMLAH_ITEM // 2, "Consumer-2"))                  # Thread konsumen kedua (mengambil 4 item)

t_p.start(); t_c1.start(); t_c2.start()  # Menyalakan semua thread secara bersamaan
t_p.join(); t_c1.join(); t_c2.join()    # Menunggu sampai seluruh proses produksi dan konsumsi benar-benar selesai

print(f"\nSisa buffer: {buffer} (harus kosong jika semua item terkonsumsi)")  # Cek akhir

[Producer] taruh item-0   (buffer: 1/3)
    [Consumer-1] ambil item-0   (buffer: 0/3)
[Producer] taruh item-1   (buffer: 1/3)
    [Consumer-2] ambil item-1   (buffer: 0/3)
[Producer] taruh item-2   (buffer: 1/3)
    [Consumer-1] ambil item-2   (buffer: 0/3)
[Producer] taruh item-3   (buffer: 1/3)
    [Consumer-2] ambil item-3   (buffer: 0/3)
[Producer] taruh item-4   (buffer: 1/3)
    [Consumer-1] ambil item-4   (buffer: 0/3)
[Producer] taruh item-5   (buffer: 1/3)
    [Consumer-2] ambil item-5   (buffer: 0/3)
[Producer] taruh item-6   (buffer: 1/3)
    [Consumer-1] ambil item-6   (buffer: 0/3)
[Producer] taruh item-7   (buffer: 1/3)
    [Consumer-2] ambil item-7   (buffer: 0/3)
[Producer] taruh item-8   (buffer: 1/3)
    [Consumer-1] ambil item-8   (buffer: 0/3)

Sisa buffer: [] (harus kosong jika semua item terkonsumsi)


LATIHAN MANDIRI

In [3]:
import threading  # Modul untuk mengelola thread

class RekeningBank:
    def __init__(self, nama, saldo_awal):
        self.nama = nama           # Menyimpan nama pemilik rekening
        self.saldo = saldo_awal    # Menyimpan saldo awal rekening
        self.lock = threading.RLock()  # Menggunakan RLock (Reentrant Lock) agar thread yang sama bisa masuk ke method berlapis

    def setor(self, jumlah):
        with self.lock:            # Mengunci rekening sebelum mengubah saldo
            self.saldo += jumlah   # Menambahkan saldo
            print(f"[{self.nama}] Setor {jumlah}, Saldo: {self.saldo}")

    def tarik(self, jumlah):
        with self.lock:            # Mengunci rekening sebelum mengubah saldo
            if self.saldo >= jumlah:       # Cek apakah uangnya cukup
                self.saldo -= jumlah       # Mengurangi saldo
                print(f"[{self.nama}] Tarik {jumlah}, Saldo: {self.saldo}")
                return True                # Mengembalikan True jika penarikan berhasil
            else:
                print(f"[{self.nama}] Tarik {jumlah} GAGAL (Saldo tidak cukup)")
                return False               # Mengembalikan False jika gagal

    def transfer(self, rekening_tujuan, jumlah):
        with self.lock:            # Mengunci rekening asal (dirinya sendiri) untuk proses transfer
            print(f"[{self.nama}] Memulai transfer {jumlah} ke {rekening_tujuan.nama}...")
            # Di sinilah RLock berperan! Method tarik() akan mengunci self.lock LAGI.
            # RLock mengizinkan ini, sedangkan Lock biasa akan membuat sistem freeze di sini (self-deadlock).
            if self.tarik(jumlah): 
                rekening_tujuan.setor(jumlah)  # Menyetorkan uang ke rekening tujuan
                print(f"[{self.nama}] Transfer berhasil ke {rekening_tujuan.nama}")

# ---- PENGUJIAN ----
rek_A = RekeningBank("Andi", 1000)
rek_B = RekeningBank("Budi", 500)

# Menjalankan transfer di dalam thread
t = threading.Thread(target=rek_A.transfer, args=(rek_B, 300))
t.start()
t.join()

[Andi] Memulai transfer 300 ke Budi...
[Andi] Tarik 300, Saldo: 700
[Budi] Setor 300, Saldo: 800
[Andi] Transfer berhasil ke Budi


In [2]:
import subprocess
import sys

# Skrip ini memuat dua skenario: deadlock dan versi perbaikannya
skrip_tiga_lock = """\n""".join([
    "import threading, time",
    "lock_a = threading.Lock()",
    "lock_b = threading.Lock()",
    "lock_c = threading.Lock()",
    "",
    "def thread1():",
    "    with lock_a:",
    "        time.sleep(0.1)",
    "        with lock_b: print('T1 selesai')",
    "",
    "def thread2():",
    "    with lock_b:",
    "        time.sleep(0.1)",
    "        with lock_c: print('T2 selesai')",
    "",
    "def thread3_deadlock():",
    "    with lock_c:",                   # Mengambil C lalu menunggu A (Memicu Circular Wait)
    "        time.sleep(0.1)",
    "        with lock_a: print('T3 selesai')",
    "",
    "def thread3_fix():",
    "    with lock_a:",                   # PERBAIKAN: Selalu ambil dari urutan abjad (A dulu, baru C)
    "        time.sleep(0.1)",
    "        with lock_c: print('T3 selesai')",
    "",
    "import sys",
    "mode = sys.argv[1] if len(sys.argv) > 1 else 'deadlock'",
    "target_t3 = thread3_deadlock if mode == 'deadlock' else thread3_fix",
    "",
    "t1 = threading.Thread(target=thread1)",
    "t2 = threading.Thread(target=thread2)",
    "t3 = threading.Thread(target=target_t3)",
    "t1.start(); t2.start(); t3.start()",
    "t1.join(); t2.join(); t3.join()",
    "print(f'STATUS: {mode.upper()} berhasil dilewati!')"
])

# Simpan skrip ke file
with open("demo_3lock.py", "w") as f:
    f.write(skrip_tiga_lock)

# 1. Menguji versi Deadlock
print("--- MENGUJI VERSI DEADLOCK ---")
try:
    hasil_macet = subprocess.run([sys.executable, "demo_3lock.py", "deadlock"], capture_output=True, text=True, timeout=2)
except subprocess.TimeoutExpired:
    print("TIMEOUT 2 DETIK: Terjadi Deadlock 3 Arah!")

# 2. Menguji versi Perbaikan
print("\n--- MENGUJI VERSI DIPERBAIKI ---")
hasil_lancar = subprocess.run([sys.executable, "demo_3lock.py", "fix"], capture_output=True, text=True, timeout=2)
print(hasil_lancar.stdout.strip())

--- MENGUJI VERSI DEADLOCK ---
TIMEOUT 2 DETIK: Terjadi Deadlock 3 Arah!

--- MENGUJI VERSI DIPERBAIKI ---
T2 selesai
T1 selesai
T3 selesai
STATUS: FIX berhasil dilewati!


In [1]:
import threading
import time
import random

def uji_buffer(ukuran):
    buffer = []
    mutex = threading.Lock()
    slot_kosong = threading.Semaphore(ukuran)   # Semaphore dibatasi sesuai ukuran (1, 3, atau 10)
    slot_terisi = threading.Semaphore(0)

    def producer(id_p):
        for i in range(3):                      # Masing-masing produsen membuat 3 barang
            item = f"P{id_p}-{i}"
            time.sleep(random.uniform(0.01, 0.05))
            slot_kosong.acquire()               # Minta slot kosong
            with mutex:
                buffer.append(item)
                print(f"[Prod-{id_p}] Taruh {item} (Isi: {len(buffer)}/{ukuran})")
            slot_terisi.release()               # Info ada barang masuk

    def consumer(id_c):
        for _ in range(3):                      # Masing-masing konsumen mengambil 3 barang
            slot_terisi.acquire()               # Minta barang
            with mutex:
                item = buffer.pop(0)
                print(f"   [Cons-{id_c}] Ambil {item} (Isi: {len(buffer)}/{ukuran})")
            slot_kosong.release()               # Info ada slot kosong

    # Membuat 2 Producer dan 2 Consumer
    threads = [
        threading.Thread(target=producer, args=(1,)),
        threading.Thread(target=producer, args=(2,)),
        threading.Thread(target=consumer, args=(1,)),
        threading.Thread(target=consumer, args=(2,))
    ]
    
    print(f"\n=== MENGUJI DENGAN UKURAN_BUFFER = {ukuran} ===")
    for t in threads: t.start()
    for t in threads: t.join()

uji_buffer(1)
uji_buffer(3)
uji_buffer(10)


=== MENGUJI DENGAN UKURAN_BUFFER = 1 ===
[Prod-2] Taruh P2-0 (Isi: 1/1)
   [Cons-1] Ambil P2-0 (Isi: 0/1)
[Prod-1] Taruh P1-0 (Isi: 1/1)
   [Cons-2] Ambil P1-0 (Isi: 0/1)
[Prod-2] Taruh P2-1 (Isi: 1/1)
   [Cons-1] Ambil P2-1 (Isi: 0/1)
[Prod-1] Taruh P1-1 (Isi: 1/1)
   [Cons-2] Ambil P1-1 (Isi: 0/1)
[Prod-2] Taruh P2-2 (Isi: 1/1)
   [Cons-1] Ambil P2-2 (Isi: 0/1)
[Prod-1] Taruh P1-2 (Isi: 1/1)
   [Cons-2] Ambil P1-2 (Isi: 0/1)

=== MENGUJI DENGAN UKURAN_BUFFER = 3 ===
[Prod-2] Taruh P2-0 (Isi: 1/3)
   [Cons-1] Ambil P2-0 (Isi: 0/3)
[Prod-1] Taruh P1-0 (Isi: 1/3)
   [Cons-2] Ambil P1-0 (Isi: 0/3)
[Prod-2] Taruh P2-1 (Isi: 1/3)
   [Cons-1] Ambil P2-1 (Isi: 0/3)
[Prod-1] Taruh P1-1 (Isi: 1/3)
   [Cons-2] Ambil P1-1 (Isi: 0/3)
[Prod-1] Taruh P1-2 (Isi: 1/3)
   [Cons-1] Ambil P1-2 (Isi: 0/3)
[Prod-2] Taruh P2-2 (Isi: 1/3)
   [Cons-2] Ambil P2-2 (Isi: 0/3)

=== MENGUJI DENGAN UKURAN_BUFFER = 10 ===
[Prod-2] Taruh P2-0 (Isi: 1/10)
   [Cons-1] Ambil P2-0 (Isi: 0/10)
[Prod-1] Taruh P1-0 (Isi: 